# TP 01 | Quels lots contrôler en priorité ? (corrigé)

**Chapitre 1 : introduction au machine learning.**

![Nervure Composites](../figures/logo-nervure.svg)

![Vue de l’atelier de Nervure Composites](../figures/atelier-nervure.jpg)

Chez **Nervure Composites**, une usine fictive, deux lignes de presses fabriquent des panneaux pour véhicules utilitaires. Un mauvais collage entre deux couches peut échapper au contrôle visuel. À la sortie de la presse, Camille, responsable qualité, connaît la ligne, la température, la pression et la durée du cycle. Elle dispose de 20 créneaux d’inspection renforcée pour 100 lots. Le verdict du laboratoire arrive huit heures plus tard.

![Coupe de la presse des lignes A et B : plateau mobile, couches empilées et panneau assemblé.](../figures/atelier-nervure.svg)

**Votre mission.** À partir de 720 lots synthétiques, rédigez pour Camille une fiche qui précise quelles mesures utiliser pour classer les lots à inspecter, quelles données vérifier et comment éviter d’utiliser une information arrivée trop tard.

Vous aurez besoin de savoir lire un tableau et de connaître les variables Python. Les opérations pandas sont introduites au fil des questions. Dans VS Code, choisissez le noyau Python du dossier `.venv`. Exécutez les cellules dans l’ordre avec **Maj + Entrée**. Modifiez les cellules de code indiquées et double-cliquez sur les blocs « Votre réponse » pour y écrire vos conclusions. Enregistrez votre notebook complété.

1. Décider quelles informations sont utilisables au moment de l’inspection.
2. Choisir une famille de méthodes et organiser l’apprentissage.
3. Diagnostiquer les mesures et proposer un traitement.
4. Rédiger une recommandation pour l’équipe qualité.

Après la fiche de cadrage, quatre défis indépendants permettent d’explorer les différences entre presses, les dossiers incomplets, la température extrême et les contrôles sélectifs. Choisissez celui qui répond à votre question.

### Qu’est-ce que le machine learning ?

Le **machine learning (ML)**, ou **apprentissage automatique**, regroupe des méthodes qui utilisent des données pour ajuster un modèle. Un modèle est ici une règle de calcul qui transforme les mesures d’un lot en un score de risque.

![Apprendre sur des lots passés puis calculer le score d’un nouveau lot](../figures/concept-apprentissage.svg)

Le verdict des lots passés sert à ajuster la règle. Camille peut ensuite appliquer cette règle à un lot dont le verdict arrivera plus tard.

Prenons une règle simple : « envoyer le lot en inspection si sa température dépasse un seuil ». Camille pourrait fixer ce seuil à partir de son expérience. Pour **apprendre** le seuil, on procède autrement : on essaie plusieurs valeurs sur des lots anciens, on compare les alertes obtenues aux verdicts du laboratoire et on retient la valeur qui réduit le mieux le coût des erreurs. C’est l’**entraînement**. Le seuil est un exemple de ce que le modèle peut apprendre ; la limite de 20 inspections pour 100 lots reste une contrainte fixée par l’usine.

Une fois la règle ajustée, on peut l’appliquer aux mesures d’un nouveau lot avant de connaître son verdict : c’est la **prédiction**. Pour savoir si elle est utile à Camille, il faudra la tester sur des lots qui n’ont pas servi à choisir le seuil.

## 1. Reconstituer ce que sait l’usine au moment de décider

**Atelier 1.1 du support MLB.** Sections 1 à 2.

![Coupe de deux panneaux : collage continu et zone décollée entre deux couches.](../figures/coupe-panneau.svg)

Une **observation** est l’unité sur laquelle porte une prédiction. Ici, une ligne du fichier représente un lot. Une **variable d’entrée** (*feature*) décrit ce lot avant la décision. La **cible** (*label*) est la réponse que l’on aimerait prédire. Pour apprendre, on utilise des lots passés dont on connaît désormais le verdict. Au moment de prédire sur un nouveau lot, ce verdict est encore inconnu.

| Moment | Colonnes du fichier | Signification |
|---|---|---|
| Fin de fabrication, instant de décision | `id`, `date` | Identifiant et date du lot. La simulation place la décision à minuit. |
| Même instant | `temperature`, `pression`, `duree`, `ligne` | Trois mesures numériques et la ligne de production A ou B. Une lettre est une catégorie, sans ordre de grandeur. |
| Huit heures après | `defaut`, `controle_final`, `label_disponible` | Verdict binaire (1 = défaut, 0 = conforme), décision finale du laboratoire et horodatage du verdict. |

Les 720 lots du CSV sont synthétiques. Les dates et la cadence ont été simplifiées. Ouvrez le fichier ci-dessous : un **DataFrame** pandas est un tableau dont les colonnes portent des noms. `head(3)` affiche les trois premières observations.

In [ ]:
# Charger les lots historiques avec leurs dates pour situer la décision et le verdict.
from pathlib import Path

import pandas as pd
from IPython.display import display

racine = Path.cwd()
if racine.name == "corrections":
    racine = racine.parent
lots = pd.read_csv(
    racine / "donnees" / "lots_train.csv", parse_dates=["date", "label_disponible"]
)
print(f"{len(lots)} lots, {len(lots.columns)} colonnes")
display(lots.head(3))

### Réceptionner un export envoyé deux fois

![Pressions de deux lots avant et après retransmission.](../figures/tp01-export-renvoye.svg)

Comparez les deux envois de chaque lot. Pour lequel la suppression d’une ligne pourrait-elle faire disparaître une information ? La figure montre un extrait des huit lignes de la simulation. Les pressions y sont arrondies pour la lecture.

Le système peut retransmettre un lot après une coupure. Une **clé** identifie une observation, ici le numéro du lot. Deux lignes identiques peuvent être deux copies du même enregistrement. Si deux lignes portent la même clé et des pressions différentes, il faut retrouver la mesure de référence.

Sur une copie des six premiers lots, on simule ces deux incidents. **Avant de coder**, prévoyez le nombre de lots réellement produits. Puis 1) affichez toutes les lignes partageant un identifiant avec `duplicated("id", keep=False)`, 2) retirez les copies strictement identiques avec `drop_duplicates()`, 3) identifiez le conflit restant. Expliquez le risque de conserver arbitrairement la dernière ligne.

In [ ]:
# Simuler deux retransmissions pour distinguer une copie exacte d’une mesure contradictoire.
extrait = lots.head(6).copy()
rectification = extrait.iloc[[1]].assign(pression=9.0)
livraison = pd.concat([extrait, extrait.iloc[[0]], rectification], ignore_index=True)
display(livraison[["id", "pression"]])
# Calculez le nombre de lignes et le nombre de clés distinctes.
# Recherchez ensuite les doublons avant et après déduplication exacte.
print(len(livraison), livraison["id"].nunique())
display(livraison.loc[livraison.duplicated("id", keep=False), ["id", "pression"]])
uniques = livraison.drop_duplicates()
conflits = uniques.loc[uniques.duplicated("id", keep=False)]
display(conflits[["id", "pression"]])
assert len(uniques) == 7 and set(conflits["id"]) == {"L0001"}

**Décision de réception.** Le fichier contient huit lignes pour six lots. La copie de L0000 répète toutes ses valeurs et peut être éliminée. Pour L0001, la pression initiale et la valeur 9,0 restent concurrentes après ce nettoyage. Je conserve les deux versions en quarantaine et demande l’horodatage ou le journal du capteur. L’ordre d’arrivée dans cet export ne garantit pas que la dernière valeur soit correcte.

### Comment passer du tableau à un problème mathématique ?

Prenons le lot L0000, que vous venez d’afficher. On peut résumer ce que sait l’usine à minuit par un vecteur dont chaque ligne correspond à une entrée.

$$
x_0 = \begin{pmatrix}\text{température}\\ \text{pression}\\ \text{durée}\\ \text{ligne}\end{pmatrix}_{\!0} = \begin{pmatrix}71{,}52\\2{,}69\\66{,}00\\\mathrm{B}\end{pmatrix}
$$

Un **vecteur** regroupe ici les valeurs d’entrée d’un seul lot, dans un ordre fixé. Le matin, le laboratoire ajoute $y_0=0$, où 0 signifie « conforme ». L’indice 0 désigne le même lot des deux côtés. Sur l’historique, on dispose ainsi de couples $(x_i,y_i)$. Le tableau des entrées est souvent noté $X$, la colonne des verdicts $y$.

On cherche une fonction $f$ telle que $\hat y_i=f(x_i)$ soit utile pour un **nouveau** lot. Le chapeau sur $\hat y$ signifie « prédiction ». Il rappelle qu’à minuit personne ne connaît encore $y_i$.

![Les mesures sont disponibles à minuit ; le verdict du laboratoire arrive à 8 h.](../figures/chronologie.svg)

Cette séparation explique pourquoi une colonne peut être précieuse pour **évaluer** une prédiction tout en étant inutilisable pour la **produire**.

Pour lire le code pandas qui suit, gardez le lot comme unité de lecture. Une ligne du tableau représente un lot. Une colonne comme `defaut` contient le même renseignement pour tous les lots. Dans `lots[lots.defaut == 1]`, la comparaison donne une série de vrai ou faux, puis les crochets gardent les lignes vraies. Ce filtre permet de retrouver des défauts **après** le retour du laboratoire. Comment retrouver les lots de la ligne B si vous êtes encore à la sortie de presse ?

**À faire.** Sur la première ligne, relevez l’heure de décision et celle du verdict. Puis désignez la cible, les entrées candidates et les colonnes à garder seulement pour assurer la traçabilité. Un collègue propose d’utiliser `controle_final` car il prédit très bien `defaut`. Expliquez ce qui se passerait sur le prochain lot.

**Indice.** Masquez mentalement tout ce qui se trouve dans la dernière ligne du tableau chronologique au moment où vous décidez. Une **fuite de données** se produit lorsque l’entraînement exploite une information qui serait indisponible lors d’une vraie prédiction.

**Votre réponse**

Pour L0000, on décide le 1er janvier 2025 à 00 h ; le verdict arrive à 08 h. La cible est `defaut`. Les quatre entrées candidates sont la température, la pression, la durée et la ligne. Je conserve l’identifiant pour retrouver le lot et la date pour respecter la chronologie. L’heure du verdict sert à vérifier quand une étiquette devient exploitable pour entraîner le modèle.

`controle_final` est un autre codage du résultat du laboratoire. Si on l’utilise pendant l’entraînement, le modèle peut sembler excellent tout en demandant un renseignement impossible à fournir au moment de décider.

**Deux erreurs, deux conséquences.** Appeler « positif » un lot signalé pour inspection permet de nommer les erreurs. Un **faux positif** est un lot conforme que l’on signale. Un **faux négatif** est un lot défectueux que l’on laisse passer. Camille dispose de 20 créneaux de contrôle renforcé pour 100 lots. Que coûte chacune de ces erreurs ? Pourquoi un score de risque pourrait-il être plus utile qu’une simple lettre « oui » ou « non » ?

![Cent lots pour vingt places en inspection renforcée.](../figures/selection-inspection.svg)

**Votre réponse**

Un faux positif mobilise l’un des 20 créneaux. Un faux négatif peut laisser un défaut atteindre le client. Un score permet de classer les lots puis d’inspecter ceux qui semblent les plus risqués, dans la limite de la capacité disponible. Un score élevé indique une priorité, il ne prouve pas un défaut.

### Au poste de Camille : deux places pour dix lots

Pour cette simulation, dix lots fictifs sont prêts en même temps. Deux places sont libres en inspection renforcée, soit la même capacité que 20 sur 100. Cette file est un exemple distinct du CSV, où les lots sont datés un par un. Le service qualité fournit un indice sur 100 : plus il est élevé, plus il recommande un contrôle. Il sert à classer les lots. Un indice de 80 ne garantit pas 80 % de probabilité de défaut.

![Indice des dix lots, affichés dans leur ordre d’arrivée.](../figures/file-controle.svg)

**Avant de coder**, relevez les deux identifiants à transmettre à Camille. Combien de créneaux réclamerait la consigne « inspecter tout indice supérieur ou égal à 70 » ? Comparez cette règle avec celle qui retient les deux premiers du classement.

In [ ]:
# Reproduire la file d’attente de Camille pour comparer priorité et ordre d’arrivée.
file = pd.DataFrame(
    {
        "lot": [f"P{i:02}" for i in range(1, 11)],
        "indice": [42, 76, 18, 88, 51, 67, 35, 81, 60, 24],
    }
)
places = 2
candidats = file.loc[file["indice"] >= 70]
prioritaires = file.nlargest(places, "indice")
print("Lots au seuil :", candidats["lot"].tolist())
display(prioritaires)
assert prioritaires["lot"].tolist() == ["P04", "P08"] and len(candidats) == 3

**Décision de Camille.** Le seuil signale P02, P04 et P08, alors que l’équipe dispose de deux places. P04 (88) et P08 (81) sont prioritaires ; P02 (76) attend. En cas d’égalité pour la dernière place, on peut choisir le lot le plus ancien selon une règle convenue à l’avance. Seuls les verdicts permettront de compter les défauts effectivement interceptés. La libération d’un lot reste une décision de l’équipe qualité.

## 2. Choisir ce que le modèle doit apprendre

Un **modèle** est une fonction qui transforme les entrées d’un lot en sortie. En apprentissage **supervisé**, on lui fournit des exemples avec la réponse attendue. Prédire une catégorie comme « défaut » est une **classification**. Prédire une mesure comme la résistance est une **régression**. En apprentissage **non supervisé**, on cherche une structure sans fournir de cible, par exemple des groupes de profils de pression semblables. En apprentissage **par renforcement**, un agent choisit des actions successives et apprend à partir de leurs conséquences, mesurées par une récompense.

**À faire.** Pour chacun des quatre besoins, choisissez une méthode. Précisez la sortie produite et le signal qui permettrait d’apprendre.

1. Anticiper le verdict du laboratoire.
2. Estimer une résistance mécanique mesurée après cuisson.
3. Regrouper des cycles de fabrication semblables, sans verdict.
4. Choisir à chaque étape un réglage qui change la suite du cycle et la qualité finale.

**Votre réponse**

1. **Classification supervisée.** La sortie est une classe. Les inspections historiques fournissent les classes de référence.
2. **Régression supervisée.** La sortie est une valeur numérique. Il faudrait collecter les résistances mesurées, absentes de notre fichier.
3. **Regroupement non supervisé.** On cherche des groupes à partir des distances entre mesures. Le numéro de groupe ne révèle pas à lui seul si un lot est défectueux.
4. **Renforcement.** La sortie est une action. Il faudrait des trajectoires avec actions et récompenses ou un simulateur. Un historique de verdicts seuls ne suffit pas.

### Que fait concrètement un algorithme pendant l’entraînement ?

Prenons quatre lots fictifs pour lesquels on connaît la température et le verdict. On limite volontairement la famille de modèles à une question : « la température atteint-elle le seuil $t$ ? ». Si oui, le modèle prédit 1, sinon 0. La forme de la règle est fixée. **Apprendre**, ici, consiste à choisir la valeur de $t$ à partir des exemples.

![Les mêmes quatre lots face à deux seuils : à 69 °C, une fausse alerte ; à 75 °C, un défaut manqué.](../figures/seuils.svg)

Les symboles donnent le verdict réel et la zone verte indique les lots signalés par chaque règle. À 69 °C, on signale à tort le lot à 73 °C. À 75 °C, on manque le défaut du lot à 70 °C. Les deux règles font **une erreur sur quatre**, mais elles n’entraînent pas la même décision industrielle.

La **perte** chiffre la conséquence d’une prédiction sur un lot. Avec la perte 0-1, une erreur vaut 1 et une bonne réponse vaut 0. Le **risque empirique** est la perte moyenne sur les $n$ lots observés :

$$
\begin{aligned}
\widehat R(t) &= \frac{1}{n}\sum_{i=1}^{n}\mathbf 1[\hat y_i(t)\ne y_i] \\[5pt]
&= \frac{1}{4} = 25\,\% \quad\text{pour chacun des deux seuils.}
\end{aligned}
$$

Le symbole $\mathbf 1$ vaut 1 si la condition entre crochets est vraie, 0 sinon. Si l’usine attribue un coût de 5 à un défaut manqué et de 1 à une fausse alerte, les coûts totaux deviennent respectivement 1 et 5. Exécutez le calcul ci-dessous, puis inversez les deux coûts. Avant de relancer, prévoyez quel seuil sera préféré.

In [ ]:
# Comparer les conséquences de deux seuils sur les mêmes quatre lots étiquetés.
temperatures_exemple = [67, 70, 73, 76]
verdicts_exemple = [0, 1, 0, 1]
cout_alerte, cout_defaut_manque = 1, 5

for seuil in [69, 75]:
    fausses_alertes = 0
    defauts_manques = 0
    for temperature, verdict in zip(
        temperatures_exemple, verdicts_exemple, strict=True
    ):
        signaler = temperature >= seuil
        fausses_alertes += int(signaler and verdict == 0)
        defauts_manques += int(not signaler and verdict == 1)
    cout = cout_alerte * fausses_alertes + cout_defaut_manque * defauts_manques
    print(
        f"Seuil {seuil} °C : {fausses_alertes} fausse(s) alerte(s), "
        f"{defauts_manques} défaut(s) manqué(s), coût total {cout}"
    )

**Après inversion des coûts.** Le seuil de 69 °C coûte maintenant 5 pour sa fausse alerte. Le seuil de 75 °C coûte 1 pour son défaut manqué. Le classement s’inverse alors que les prédictions restent identiques. Le choix dépend donc de la fonction de perte. Quatre lots permettent de vérifier le mécanisme, mais ils sont trop peu nombreux pour choisir un seuil de production.

### Pourquoi séparer des lots que l’on connaît déjà ?

Imaginez une règle qui mémorise le verdict de chaque identifiant. Elle pourrait restituer les quatre réponses sans erreur. Pour un cinquième identifiant, sa table ne donne aucune réponse. C’est un cas extrême de **surapprentissage** : les résultats sur les exemples utilisés pour ajuster la règle surestiment sa capacité à **généraliser**, c’est-à-dire à prédire sur de nouveaux exemples.

Pour reproduire l’usage en usine, on avance dans le temps. La période la plus ancienne sert à apprendre. Une période suivante sert à comparer les choix. La dernière période reste réservée jusqu’à la décision finale. Cette discipline concerne aussi la préparation des données : si une pression manque en janvier, utiliser la médiane calculée jusqu’en décembre introduirait dans le passé des informations venues du futur.

![Les lots anciens servent à ajuster le modèle, les suivants à choisir les réglages et les plus récents au test final.](../figures/decoupage.svg)

**Du problème à l’évaluation.** Un *paramètre* est appris à partir des exemples, comme le seuil de température choisi par un arbre pour séparer deux groupes. Un *hyperparamètre* est un réglage du procédé d’apprentissage, comme la profondeur maximale autorisée pour cet arbre. On ajuste les paramètres sur le jeu d’entraînement, on compare les réglages sur un jeu de validation puis on évalue le choix final sur un jeu de test réservé.

**À faire.** Remettez ces opérations dans l’ordre et justifiez la place du test : A. évaluer sur les lots réservés ; B. définir la décision ; C. ajuster le modèle sur les lots anciens ; D. comparer les réglages sur les lots de validation ; E. séparer chronologiquement les lots. Vérifiez aussi qu’à la date d’entraînement le verdict des lots retenus est déjà arrivé.

**Votre réponse**

B, E, C, D, A. La question métier fixe l’instant de décision. On sépare les périodes avant d’ajuster quoi que ce soit. Le test reproduit une utilisation future. Choisir la profondeur après avoir lu son score sur le test ferait de celui-ci un second jeu de validation.

## 3. Examiner les mesures avant d’entraîner

**Atelier 1.2 du support MLB.** Sections 3 à 4.

**Commencez par compter.** Le nombre de lignes et le nombre d’identifiants distincts doivent coïncider si chaque lot apparaît une seule fois. `nunique()` compte les valeurs distinctes. `value_counts()` compte le nombre d’occurrences de chaque valeur. Complétez la cellule pour obtenir le nombre de lots défectueux et leur proportion. La proportion est le nombre de défauts divisé par le nombre total de lots.

In [ ]:
# Vérifier combien de lots distincts sont présents et quelle part présente un défaut.
print("Identifiants distincts :", lots["id"].nunique())
print(lots["defaut"].value_counts().sort_index())
proportion = lots["defaut"].mean()
print(f"Proportion de défauts : {proportion:.1%}")
assert lots["id"].nunique() == 720 and lots["defaut"].sum() == 192

La classe 1 représente 192 lots sur 720, soit 26,7 %. Comme la cible vaut 0 ou 1, sa somme compte les défauts et sa moyenne donne directement leur proportion. Il s’agit de la fréquence dans cet historique.

### Que peut-on réellement déduire d’un capteur silencieux ?

Sur cinq lots d’exemple, on lit les pressions $[2{,}7;\ 2{,}9;\ ?;\ 3{,}1;\ 3{,}3]$. Une des cinq pressions est absente.

$$
\text{Taux d’absence} = \frac{1}{5} = 20\,\%
$$

Pour estimer la **médiane**, on trie les quatre valeurs connues et on prend le milieu des deux valeurs centrales.

$$
\begin{aligned}
\text{Médiane} &= \frac{2{,}9+3{,}1}{2} \\[5pt]
&= 3{,}0
\end{aligned}
$$

On peut remplacer la case vide par 3,0 pour fournir une entrée numérique au modèle. Ce nombre décrit le centre des mesures connues. La pression réelle de ce lot demeure inconnue.

Pourquoi conserver aussi l’indicateur « capteur silencieux » ? Si le silence est dû à une coupure aléatoire, les lots incomplets peuvent ressembler aux autres. Si le capteur décroche surtout à forte pression, les valeurs enregistrées sous-représentent les hautes pressions. Remplir toutes les cases par 3,0 peut alors masquer un risque. Compter les absences, les comparer entre lignes et vérifier le fonctionnement du capteur répondent à trois questions différentes.

![Les quatre points bleus représentent les mesures observées. Le losange vide situe la médiane entre les deux mesures centrales.](../figures/mediane.svg)

**Pression absente.** Une case vide signifie ici que la mesure n’a pas été enregistrée. Mettre zéro inventerait une pression réellement mesurée à zéro. Dans pandas, `isna()` repère les valeurs manquantes et la somme des booléens compte les cases vraies. Calculez le nombre et le pourcentage de pressions absentes. Proposez une stratégie qui permettrait de traiter un nouveau lot incomplet.

**Indice.** La médiane est la valeur centrale des mesures triées (la moyenne des deux valeurs centrales si l’effectif est pair). Elle est moins sensible aux valeurs extrêmes que la moyenne. Remplacer une valeur absente par une estimation s’appelle l’**imputation**.

In [ ]:
# Repérer les silences du capteur avant de choisir une valeur de remplacement.
pression = lots["pression"]
manque = pression.isna()
print("Exemples de lots concernés :", lots.loc[manque, "id"].head(3).tolist())
print(f"{manque.sum()} absences sur {len(lots)}, soit {manque.mean():.1%}")
mediane = pression.median()
pression_preparee = pression.fillna(mediane)
print(f"Médiane : {mediane:.3f}")
assert manque.sum() == 19 and pression_preparee.notna().all()

**Votre réponse**

19 pressions sur 720 manquent, soit 2,6 %. On peut conserver la ligne, remplacer la pression par la médiane du train (2,999 environ) et conserver un indicateur de mesure absente. On réutilisera cette même médiane sur les futurs lots. L’imputation ne répare pas le capteur et ne rétablit pas la vraie pression.

**Température hors plage.** Pour ce scénario, le procédé est annoncé entre 40 et 110 °C, bornes incluses. Une **valeur aberrante** est une observation inhabituelle selon une référence précisée. Un signal hors plage peut refléter une panne du capteur ou une surchauffe réelle.

![Température des 720 lots. Les deux lignes pointillées délimitent la plage du procédé ; un point isolé la dépasse.](../figures/temperatures.svg)

Repérez la mesure isolée sur le graphique, puis retrouvez le lot dans le fichier. Complétez la cellule pour afficher les identifiants et températures hors plage. `between(40, 110)` fournit un masque vrai dans la plage ; `~` inverse le masque ; `.loc[masque, colonnes]` sélectionne les lignes et colonnes demandées. Choisissez ensuite une action avant toute suppression.

In [ ]:
# Isoler les mesures qui dépassent les limites physiques annoncées pour le procédé.
dans_plage = lots["temperature"].between(40, 110)
hors_plage = lots.loc[~dans_plage, ["id", "temperature"]]
display(hors_plage)
assert hors_plage["id"].tolist() == ["L0010"]

**Votre réponse**

L0010 affiche 150 °C. Je mets la mesure en vérification, conserve la valeur brute et demande le journal du capteur ou une mesure indépendante. Supprimer d’office ce lot pourrait effacer précisément l’incident que l’on souhaite détecter.

### Auditer une variable qui prédit presque trop bien

Un collègue propose d’utiliser `controle_final` pour repérer les défauts. Un **tableau croisé** compte combien de lots possèdent chaque combinaison de deux valeurs. Construisez-le avec `pd.crosstab` pour comparer le contrôle final au verdict binaire. Quel résultat obtiendrait une règle qui annonce un défaut dès que le lot est rejeté ? Reconstituez ensuite les deux heures du lot L0000. Décidez si la règle pourrait fonctionner en atelier.

In [ ]:
# Examiner quand le contrôle final devient connu avant de l’utiliser comme information.
display(lots[["id", "date", "label_disponible", "controle_final", "defaut"]].head())
# Croisez controle_final (lignes) et defaut (colonnes).
# Comparez la règle déduite du tableau au verdict sur les 720 lots.
table = pd.crosstab(lots["controle_final"], lots["defaut"])
display(table)
rejets = table.index[table[1] > 0]
accord = lots["controle_final"].isin(rejets).eq(lots["defaut"]).mean()
print(f"Accord sur l’historique : {accord:.0%}")
assert accord == 1

**Conclusion.** Les 528 lots conformes et les 192 lots défectueux se rangent dans deux catégories distinctes du contrôle final. L’accord de 100 % s’explique par la construction du fichier. À minuit pour L0000, la presse a produit ses mesures. Le contrôle du laboratoire arrive à 8 h. Utiliser cette colonne donnerait au modèle une copie indirecte de la réponse, une fuite de cible. Je garde le contrôle final pour l’audit après inspection et je l’exclus des entrées du modèle.

### Rejouer la prise de poste du 29 janvier

À minuit le 29 janvier 2025, L0028 sort de presse avec une pression absente. Le journal mentionne également L0010, un ancien lot dont la température était hors plage. Camille demande deux choses : comment traiter le dossier courant, et quels verdicts passés peut-on déjà consulter ?

Dans un export historique, valeurs et réponses sont réunies. Reconstituez le registre tel qu’il existait au poste : comparez la date de production **et** la date de disponibilité du verdict à l’instant de décision. En Python, la conjonction de deux masques pandas s’écrit avec & et des parenthèses autour de chaque comparaison.

In [ ]:
# Reconstituer les informations disponibles à la prise de poste du 29 janvier.
instant = pd.Timestamp("2025-01-29 00:00")
dossier = lots.loc[lots["id"] == "L0028", ["id", "date", "ligne", "pression"]]
display(dossier)
produits = lots["date"] <= instant
connus = produits & (lots["label_disponible"] <= instant)
print(f"{produits.sum()} lots produits, {connus.sum()} verdicts connus")
anciens = lots.loc[connus]
display(anciens.loc[anciens["id"] == "L0010", ["id", "temperature", "defaut"]])
assert produits.sum() == 29 and connus.sum() == 28
assert "L0028" not in anciens["id"].values

**Note de relève.** À minuit, 29 lots ont été produits et 28 verdicts sont disponibles. L0028 vient de A. Je conserve la pression comme manquante, je vérifie le journal du capteur et j’attends 8 h pour son verdict. Pour simuler une imputation à cette date, la médiane doit venir du passé. Celle des 720 lots explorés plus haut inclut des mesures postérieures.

Pour L0010, le laboratoire a déjà confirmé un défaut et le capteur a enregistré 150 °C. Cette coïncidence justifie une enquête sur le capteur et le procédé. Elle ne prouve ni la cause du défaut ni un décollement précis : le fichier enregistre un verdict binaire, sans détailler la défaillance.

## 4. Remettre votre fiche de cadrage

En cinq lignes, indiquez la décision à prendre, la cible et son délai, les entrées autorisées, les deux contrôles qualité avec leurs résultats chiffrés, puis ce qu’il reste à vérifier avant d’utiliser un modèle. Un collègue doit pouvoir comprendre votre recommandation en lisant uniquement cette fiche.

Adressez la fiche à Camille, citez un identifiant de lot et précisez qui décide de sa libération.

**Votre réponse**

1. Prioriser l’inspection à la fin de fabrication, dans la limite des créneaux disponibles.
2. Prédire le défaut constaté huit heures plus tard.
3. Utiliser température, pression, durée et ligne disponibles à cet instant. Conserver les identifiants pour l’audit.
4. Documenter les 19 pressions absentes, estimer leur remplacement sur le train et vérifier L0010 à 150 °C.
5. Comparer une première règle et un modèle sur des lots ultérieurs avec le coût métier des erreurs. Ce diagnostic seul ne mesure aucune performance prédictive.

**Transmission à Camille.** L0028 reste traçable malgré sa pression absente. Le score oriente l’inspection, le laboratoire établit le verdict et l’équipe qualité conserve la décision de libération.

### Prise en main de Spark | Deux modèles lisent le même lot

Dans le terminal du dépôt, lancez `uv sync --locked --extra spark` avant la première utilisation. Le premier démarrage récupère Java. Dans cette démo, deux arbres déjà paramétrés apprennent à partir de la température des lots historiques. L’un peut poser une seule question, l’autre jusqu’à trois questions successives. Spark utilise ici votre poste. Retrouvez la transformation qui rassemble les entrées, puis celle qui applique chaque arbre. Les deux profondeurs donnent-elles toujours la même classe sur les lots de validation ?

In [ ]:
# Comparer deux profondeurs sur les mêmes lots, sans utiliser leur verdict pour prédire.
import os
import sys

import cjdk
from pyspark.ml.classification import DecisionTreeClassifier
from pyspark.ml.feature import VectorAssembler
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

os.environ["JAVA_HOME"] = str(cjdk.java_home(vendor="zulu", version="21"))
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ.setdefault("SPARK_LOCAL_IP", "127.0.0.1")
demo = SparkSession.builder.master("local[2]").appName("Nervure-accueil").getOrCreate()
try:
    assembler = VectorAssembler(inputCols=["temperature"], outputCol="features")

    def mesures_spark(fichier):
        # Les données déjà lues par pandas restent sur le poste pour cette petite démo.
        cadre = pd.read_csv(racine / "donnees" / fichier)[
            ["id", "temperature", "defaut"]
        ]
        return assembler.transform(
            demo.createDataFrame(cadre.dropna().to_dict("records"))
        )

    historique = mesures_spark("lots_train.csv")
    lecture = mesures_spark("lots_validation.csv")
    for profondeur in (1, 3):
        arbre_demo = DecisionTreeClassifier(
            labelCol="defaut",
            maxDepth=profondeur,
            seed=42,
            predictionCol=f"classe_{profondeur}",
            probabilityCol=f"proba_{profondeur}",
            rawPredictionCol=f"brut_{profondeur}",
        ).fit(historique)
        lecture = arbre_demo.transform(lecture)
    lecture.select("id", "temperature", "classe_1", "classe_3").orderBy("id").show(5)
    print(
        "Lots en désaccord :",
        lecture.filter(F.col("classe_1") != F.col("classe_3")).count(),
    )
finally:
    demo.stop()

**Corrigé.** Sur les lots de validation, les deux classes diffèrent pour 23 lots. Les colonnes classe_1 et classe_3 sont des décisions binaires sur le même panneau. VectorAssembler range la température dans le vecteur attendu par MLlib. Le premier fit apprend une coupure, le second peut affiner certaines zones. Le compteur porte sur tous les lots évaluables, y compris ceux absents des cinq premières lignes. Pour départager les modèles, il faudra confronter leurs décisions aux verdicts et au coût des erreurs au TP 02.

## Défi facultatif | Une moyenne peut-elle masquer un problème de ligne ?

Camille remarque davantage de défauts sur B. Pour comparer deux lignes de tailles différentes, rapportez les défauts au nombre de lots produits sur **chacune**. C’est la fréquence conditionnelle : on répond à la question « parmi les lots passés sur cette ligne, quelle part présente un défaut ? ».

![Part des lots avec défaut sur chaque presse, calculée sur les 720 lots.](../figures/lecture-lignes.svg)

A compte 81 défauts sur 367 lots (22,1 %) et B en compte 111 sur 353 (31,4 %). L’écart vaut environ 9,4 points de pourcentage avant arrondi. Il décrit l’historique. Il ne dit pas ce qui arriverait si l’on déplaçait le même lot de B vers A : les matières ou les réglages peuvent aussi différer. 

Comparez la fréquence de pression absente entre A et B. Pour une comparaison équitable, divisez dans chaque groupe le nombre de pressions absentes par **l’effectif de ce groupe**. Avec `assign`, ajoutez le masque des absences dans un nouveau tableau ; avec `groupby("ligne")`, effectuez le calcul séparément pour A et B. Quelle ligne vérifieriez-vous en premier ? Quelle conclusion serait prématurée ?

In [ ]:
# Comparer les absences de pression en tenant compte du volume propre à chaque presse.
par_ligne = lots.assign(absente=manque).groupby("ligne")["absente"]
bilan = par_ligne.agg(["size", "sum", "mean"])
display(bilan)
assert bilan["sum"].sum() == 19

La colonne `mean` rapporte les absences à l’effectif de chaque ligne. Je commence par B, qui présente le taux le plus élevé.

$$
\begin{aligned}
\text{Ligne A} &:\quad \frac{8}{367} \approx 2{,}2\,\% \\[5pt]
\text{Ligne B} &:\quad \frac{11}{353} \approx 3{,}1\,\%
\end{aligned}
$$

Une différence sur cet échantillon ne démontre pas une panne de cette ligne : il faut aussi vérifier les dates, les opérations de maintenance et le mode de collecte.

## Défi facultatif | Que perd-on en supprimant les lots incomplets ?

Une autre équipe propose de retirer les lots sans pression avant toute analyse. Comparez le nombre de défauts et leur fréquence pour les deux groupes. Calculez ensuite le taux de défauts que Camille annoncerait avec et sans cette suppression. Distinguez ce que vous observez de ce que vous pouvez conclure sur le capteur.

In [ ]:
# Mesurer combien de défauts seraient exclus avec les dossiers sans pression.
sans_pression = lots["pression"].isna()
# Regroupez defaut avec groupby(sans_pression), puis utilisez agg(["size", "sum", "mean"]).
# Comparez la moyenne sur tous les lots à celle des lignes complètes.
display(lots.groupby(sans_pression)["defaut"].agg(["size", "sum", "mean"]))
print(
    f"Tous : {lots.defaut.mean():.2%}, pression connue : {lots.loc[~sans_pression, 'defaut'].mean():.2%}"
)
assert lots.loc[sans_pression, "defaut"].sum() == 8

**Interprétation.** Supprimer 19 lignes ferait disparaître 8 défauts. La proportion globale passerait de 26,67 % à 26,25 %. Ce faible mouvement masque un écart entre groupes (8 sur 19 contre 184 sur 701). Le groupe incomplet est petit. Ces chiffres justifient de suivre les absences et de consulter la maintenance. Ils ne permettent pas d’attribuer un défaut à la panne du capteur.

## Défi facultatif | Une seule mesure peut-elle changer le portrait de la presse ?

Camille doit résumer la température habituelle des lots. Le capteur de L0010 affiche 150 °C. Une mesure est dite **influente** lorsqu’un résultat change sensiblement selon qu’on la prend en compte ou non. L’écart-type décrit la dispersion autour de la moyenne, dans la même unité que la mesure.

1. Avant de calculer, classez la moyenne, la médiane et l’écart-type selon leur sensibilité attendue à une valeur extrême.
2. Construisez deux colonnes de statistiques en °C, avec tous les lots puis avec les températures comprises entre 40 et 110 °C. Calculez la différence pour chaque statistique.
3. Choisissez un indicateur pour décrire le centre du procédé et un autre pour enquêter sur les anomalies. Que faut-il vérifier avant d’écarter la mesure du dossier qualité ?

In [ ]:
# Tester la sensibilité des statistiques de température au lot situé hors plage.
temperature = lots["temperature"]
plage_procede = temperature.between(40, 110)
statistiques = ["mean", "median", "std"]
# Essayez temperature.agg(statistiques), puis refaites le calcul sur temperature[plage_procede].
# Réunissez les deux séries dans un DataFrame et soustrayez les colonnes.
resume = pd.DataFrame(
    {
        "tous": temperature.agg(statistiques),
        "dans la plage": temperature[plage_procede].agg(statistiques),
    }
)
resume["écart en °C"] = resume["tous"] - resume["dans la plage"]
display(resume.round(3))
assert (~plage_procede).sum() == 1

**Correction.** La moyenne augmente de 0,111 °C et la médiane de seulement 0,001 °C environ. L’écart-type passe de 5,068 à 5,880 °C. Une seule valeur pèse davantage dans le calcul de la dispersion, qui utilise des écarts au carré. Pour le point quotidien, je présente la médiane et je suis séparément le nombre de dépassements de la plage. J’isole L0010 pour vérifier le capteur et le journal de presse. Les deux calculs décrivent des scénarios et la mesure brute reste traçable.

## Défi facultatif | Que saurait-on si le laboratoire ne contrôlait que les lots les plus chauds ?

Dans le fichier, les 720 verdicts sont connus. Imaginez maintenant que le laboratoire ait contrôlé uniquement les 20 % de lots les plus chauds. Un **biais de sélection** apparaît quand la façon de choisir les lots observés modifie la population que décrivent les résultats.

1. Construisez un masque avec le 80e percentile de température (la valeur qui sépare approximativement les 80 % les moins chauds des 20 % restants).
2. Comptez les lots et les défauts dans chaque groupe. Comparez le taux parmi les lots contrôlés au taux sur toute la production.
3. Si les verdicts des lots non contrôlés étaient réellement inconnus, quel taux pourrait-on annoncer ? Proposez un plan de contrôle pour estimer le taux global.

In [ ]:
# Simuler un contrôle réservé aux lots les plus chauds pour étudier le biais de sélection.
limite_chaleur = lots["temperature"].quantile(0.8)
controle_simule = lots["temperature"].ge(limite_chaleur)
# Avec groupby(controle_simule), calculez size, sum et mean sur defaut.
# Vérifiez le dénominateur de chaque taux avant de les comparer.
groupes = lots.groupby(controle_simule)["defaut"].agg(["size", "sum", "mean"])
display(groupes.rename(index={True: "contrôlés", False: "autres"}))
print(f"Ensemble de la production : {lots.defaut.mean():.1%}")
assert groupes.loc[True, "size"] == 144

**Correction.** Le laboratoire verrait 90 défauts sur 144 lots sélectionnés, soit 62,5 %. Le fichier complet donne 192 sur 720, soit 26,7 %. Les 102 défauts parmi les 576 autres lots sont visibles uniquement parce que nous simulons la sélection après coup. En conditions réelles, un lot sans verdict resterait de statut inconnu. Pour estimer la fréquence globale, on peut réserver des inspections à un échantillon aléatoire de toute la production et en suivre séparément les résultats.